# Merchant Ranking Based on Essential-or-not

this notebook ranks merchants within three segments based on their category: 

- essential: needed in all circumstances 
- borderline: useful but leaning more discretionary 
- not-essential: other products 

In [1]:
import sys
from functools import reduce
from datetime import timedelta

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import pyspark.sql.functions as F
from pyspark.sql.window import Window

In [2]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/08 21:39:11 WARN Utils: Your hostname, tray, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/08 21:39:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/08 21:39:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Load Merged_df

In [3]:
merged_df = spark.read.parquet("../data/curated/merged_transactions_sa2.parquet")


## Check Segments 

segments are based on company category tags.

In [4]:
merged_df.groupBy("is_essential").count().orderBy("is_essential").show()

+-------------+-------+
| is_essential|  count|
+-------------+-------+
|   borderline|2345050|
|    essential|1617656|
|not_essential|9651969|
+-------------+-------+



In [5]:
(merged_df
    .select("is_essential", "category")
    .distinct()
    .orderBy("is_essential", "category")
    .show(100, truncate=False))

+-------------+-------------------------------------------------------------------------------------+
|is_essential |category                                                                             |
+-------------+-------------------------------------------------------------------------------------+
|borderline   |cable, satellite, and other pay television and radio services                        |
|borderline   |computers, computer peripheral equipment, and software                               |
|borderline   |furniture, home furnishings and equipment shops, and manufacturers, except appliances|
|borderline   |stationery, office supplies and printing and writing paper                           |
|essential    |motor vehicle supplies and new parts                                                 |
|essential    |opticians, optical goods, and eyeglasses                                             |
|essential    |shoe shops                                                         

# Merchant Summary

In [6]:
# check nulls in merged_df
merged_df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in merged_df.columns]).show()    
# check total rows in merged_df
merged_df.count()

def merchant_summary(spark_df):
    return spark_df.groupBy("merchant_abn").agg(
        F.first("merchant_name", ignorenulls=True).alias("merchant_name"),
        F.first("category", ignorenulls=True).alias("category"),
        F.first("is_essential", ignorenulls=True).alias("is_essential"),
        F.first("take_rate", ignorenulls=True).alias("take_rate"),
        F.sum("dollar_value").alias("total_revenue"),
        F.count("order_id").alias("num_transactions"),
        F.avg("dollar_value").alias("avg_order_value"),
        F.avg("fraud_probability").alias("mean_fraud_probability"),

    )

merchant_summary_df = merchant_summary(merged_df)
merchant_summary_df.show(10, truncate=False)
merchant_summary_df.groupBy("is_essential").count().orderBy("is_essential").show()

+--------+------------+-------+--------------+------------+--------+-----------+----+-------+-----+------+-----------------+---------------------+-------------+----+--------+------------+---------+--------------+-------------+--------------+---------------+-----------------------+------------------+-----------------------------+---------------------------+------------------+-------------------------+----------------------------+-------------------------+----------------------+----------+------------+
|postcode|merchant_abn|user_id|order_datetime|dollar_value|order_id|consumer_id|name|address|state|gender|fraud_probability|is_same_day_duplicate|merchant_name|tags|category|revenue_band|take_rate|category_group|total_revenue|n_transactions|avg_order_value|avg_merchant_fraud_prob|Median_age_persons|Median_mortgage_repay_monthly|Median_tot_prsnl_inc_weekly|Median_rent_weekly|Median_tot_fam_inc_weekly|Average_num_psns_per_bedroom|Median_tot_hhd_inc_weekly|Average_household_size|prop_18_34|is

+------------+-----------------------------------+-------------------------------------------------------------------------------------+-------------+---------+------------------+----------------+------------------+----------------------+
|merchant_abn|merchant_name                      |category                                                                             |is_essential |take_rate|total_revenue     |num_transactions|avg_order_value   |mean_fraud_probability|
+------------+-----------------------------------+-------------------------------------------------------------------------------------+-------------+---------+------------------+----------------+------------------+----------------------+
|10279061213 |Pede Ultrices Industries           |computer programming, data processing, and integrated systems design services        |not_essential|5.71     |166058.19172755213|555             |299.2039490586525 |16.957578431331797    |
|10323485998 |Nunc Inc.                     

+-------------+-----+
| is_essential|count|
+-------------+-----+
|   borderline|  699|
|    essential|  612|
|not_essential| 2715|
+-------------+-----+



# Ranking within Each Segment 

percentile ranking used for comparison on same scale
ranking score uses weighted average of percentile rankings 

In [7]:
FEATURES = {
    # score name:            (metric column,            higher_is_better, weight)
    "take_rate_score":       ("take_rate",              True,  1.0),
    "dollar_value_score":    ("total_revenue",          True,  1.0),
    "transaction_score":     ("num_transactions",       True,  1.0),
    "avg_order_value_score": ("avg_order_value",        True,  1.0),

}

In [8]:
missing = [m for m, _, _ in FEATURES.values() if m not in merchant_summary_df.columns]
print("Missing from summary:", missing)
print("Summary columns:", merchant_summary_df.columns)

Missing from summary: []
Summary columns: ['merchant_abn', 'merchant_name', 'category', 'is_essential', 'take_rate', 'total_revenue', 'num_transactions', 'avg_order_value', 'mean_fraud_probability']


In [9]:

def add_percentile_scores(summary_df):
    """Percentile score per feature within each segment (1 = best)."""
    out = summary_df
    for score_col, (metric, higher_is_better, _) in FEATURES.items():
        order = F.col(metric) if higher_is_better else F.desc(metric)
        w = Window.partitionBy("is_essential").orderBy(order)
        out = out.withColumn(score_col, F.percent_rank().over(w))
    return out


def add_ranking(scored_df):
    """Weighted average of the feature scores, ranked within each segment."""
    total_w = sum(wt for _, _, wt in FEATURES.values())
    score = sum(F.col(c) * wt for c, (_, _, wt) in FEATURES.items()) / total_w
    w = Window.partitionBy("is_essential").orderBy(
        F.desc("ranking_score"), F.asc("merchant_abn")  # merchant_abn breaks ties consistently
    )
    return (
        scored_df
        .withColumn("ranking_score", score)
        .withColumn("rank", F.row_number().over(w))
    )


ranking = add_ranking(add_percentile_scores(merchant_summary_df)).cache()

In [10]:
SEGMENTS = {"essential", "borderline", "not_essential"}
for segment in SEGMENTS:
    print(f"Top merchants in {segment} segment:")
    ranking.filter(F.col("is_essential") == segment).orderBy(F.desc("ranking_score")).show(10, truncate=False)

Top merchants in essential segment:


+------------+-------------------------+----------------------------------------+------------+---------+------------------+----------------+------------------+----------------------+------------------+------------------+------------------+---------------------+------------------+----+
|merchant_abn|merchant_name            |category                                |is_essential|take_rate|total_revenue     |num_transactions|avg_order_value   |mean_fraud_probability|take_rate_score   |dollar_value_score|transaction_score |avg_order_value_score|ranking_score     |rank|
+------------+-------------------------+----------------------------------------+------------+---------+------------------+----------------+------------------+----------------------+------------------+------------------+------------------+---------------------+------------------+----+
|12771097467 |At Pretium Corp.         |motor vehicle supplies and new parts    |essential   |6.95     |4932394.026652823 |1340            |36

In [11]:
#export to csv
ranking.toPandas().to_csv("../data/curated/merchant_ranking_segmented.csv", index=False)

# Evaluate Rankings

# Notes

In [12]:
print("Ranking saved to ../data/curated/merchant_ranking_segmented.csv")

Ranking saved to ../data/curated/merchant_ranking_segmented.csv
